# PAC-Learning Validation — 1D $\mathbb{Z}_2$ Schwinger Model

## From $d = O(\log n)$ to $d = O(\text{poly}(n))$

This notebook implements the Schwinger model extension proposed in the BRGD25 slides:
the 1D $\mathbb{Z}_2$ Lattice Gauge Theory with **spatially-dependent coupling strength**.
This moves the learning problem from the $d=O(\log n)$ regime (where quantum advantage
is proven) toward the $d=O(\text{poly}(n))$ regime (where the method is expected to fail).

---

## Model

$$H = m\sum_i(-1)^i Z_{m_i} + \sum_i g_i\, X_{m_i}Z_{l_i}X_{m_{i+1}} + h\sum_i X_{l_i}$$

| Symbol | Role | Status |
|---|---|---|
| $m$ | staggered fermion mass | **known** |
| $h$ | electric field coupling | **known** |
| $g_i$ | hopping amplitude on link $i$ | **unknown** — spatially varying |
| $x$ | initial gauge field configuration $\in\{0,1\}^{N-1}$ | input variable |

**Qubit layout** for $N=3$ matter sites (5 qubits total, interlaced matter/link):
```
qubit: 0      1       2      3       4
role:  m₀  link₀₁   m₁  link₁₂   m₂
```

---

## Why this is harder — and in what sense it challenges the framework

| Property | TFIM (d=1) | Model 4 (d=2) | Schwinger (d=N−1) |
|---|---|---|---|
| Unknown params | $\beta$ (uniform X field) | $\alpha, \beta$ | $g_0, g_1, \ldots, g_{N-2}$ |
| d_params | 1 | 2 | **N−1 = poly(n)** |
| Fourier modes | $(4nr+1)$ | $(4r+1)^2$ | $(4r+1)^{N-1}$ — **exponential in N** |
| Upload Pauli | single-body $X_i$ | $ZZ$ + $X_i$ | **3-body** $X_{m_i}Z_{l_i}X_{m_{i+1}}$ |
| Input space | $2^{C(n,2)}$ graph topologies | same | $2^{N-1}$ **gauge configurations** |
| Gauge constraint | none | none | **Gauss's law** must be respected |

The 3-body Pauli upload gate means the D block computes parity over **three** qubits
(matter-link-matter), making the frequency register sensitive to the correlated
motion of fermions and gauge fields simultaneously.  The slides hypothesise that
BRGD25 will either fail at large $N$ (high-frequency Fourier content) or find
an unexpected sparse structure that keeps it tractable.

---

## Observables

| Observable | Physical meaning | Pauli string |
|---|---|---|
| Staggered magnetisation $\langle M_\mathrm{stag}\rangle$ | Fermion condensate order parameter | $\frac{1}{3}(Z_0 - Z_2 + Z_4)$ |
| Electric flux $\langle E_{01}\rangle$ | Gauge field on link 0–1 | $X_1$ (link qubit) |

## 1 — Imports & parameters

In [ ]:
import sys, os, time
sys.path.append(os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
from scipy.linalg import expm
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, SparsePauliOp
from sklearn.linear_model import Lasso

from src.quantum_routines import CircuitBuilder

# ── Model parameters ──────────────────────────────────────────────────────────
N_MATTER    = 3      # matter sites → 5 qubits (2N-1)
N_QUBITS    = 2 * N_MATTER - 1   # = 5
N_LINKS     = N_MATTER - 1       # = 2 (d_params = 2)

# Known parameters (fixed, not learned)
MASS        = 0.5    # staggered fermion mass m
H_ELEC      = 0.3   # electric field coupling h

# Unknown parameters (what the learner must infer from training data)
TRUE_G      = [1.0, 0.7]   # [g_0, g_1] — spatially-varying hopping amplitudes

# Learning setup
R_STEPS     = 2      # Trotter steps (keep small: 5-qubit circuit is expensive)
LASSO_ALPHA = 1e-5
EPSILON_B   = 0.0
T_VALUES    = np.linspace(0, 3, 13)

# Input space: gauge field configurations x ∈ {0,1}^{N-1}
# x_state = binary string of length N_LINKS ('00','01','10','11' for N=3)
ALL_X_STATES = [format(k, f'0{N_LINKS}b') for k in range(2**N_LINKS)]

# Test state: '10' (link 0 flipped, link 1 unflipped) — unseen in training
TEST_X      = '10'
TRAIN_X     = [x for x in ALL_X_STATES if x != TEST_X]  # 3 training states
NUM_TRAIN   = len(TRAIN_X)

print(f'1D Z₂ Schwinger model:  N_matter={N_MATTER},  N_qubits={N_QUBITS}')
print(f'  Known:    m={MASS},  h={H_ELEC}')
print(f'  Unknown:  g = {TRUE_G}  (d={N_LINKS} parameters)')
print(f'  Input space: {ALL_X_STATES}  (gauge field configurations)')
print(f'  Training: {TRAIN_X}  →  Test: {TEST_X}')
print(f'  r={R_STEPS} Trotter steps,  LASSO_alpha={LASSO_ALPHA}')

## 2 — Hamiltonian & reference functions

In [ ]:
def H_schwinger(mass, g_list, h_elec):
    """Full Hamiltonian for the 1D Z₂ Schwinger model.

    H = m*sum_i (-1)^i Z_{matter_i}
      + sum_i g_i * X_{m_i} Z_{l_i} X_{m_{i+1}}
      + h * sum_i X_{link_i}

    Qubit layout (interlaced, Qiskit little-endian):
      qubit 0: matter 0,  qubit 1: link 0-1,  qubit 2: matter 1,
      qubit 3: link 1-2,  qubit 4: matter 2
    """
    n = N_QUBITS
    ops, vals = [], []

    # Staggered mass: m*(-1)^i * Z on matter site i (qubit 2i)
    for i, mq in enumerate(range(0, n, 2)):
        p = ['I']*n; p[n-1-mq] = 'Z'
        ops.append(''.join(p)); vals.append((-1)**i * mass)

    # Gauge coupling: g_i * X_{m_i} Z_{l_i} X_{m_{i+1}}
    for i in range(N_LINKS):
        m_left, link, m_right = 2*i, 2*i+1, 2*i+2
        p = ['I']*n
        p[n-1-m_left] = 'X'; p[n-1-link] = 'Z'; p[n-1-m_right] = 'X'
        ops.append(''.join(p)); vals.append(g_list[i])

    # Electric field: h * X on each link qubit
    for i in range(N_LINKS):
        lq = 2*i+1
        p = ['I']*n; p[n-1-lq] = 'X'
        ops.append(''.join(p)); vals.append(h_elec)

    return SparsePauliOp(ops, vals)


# ── Observables ───────────────────────────────────────────────────────────────
# Staggered magnetisation: (1/N_matter) * sum_i (-1)^i <Z_{matter_i}>
# = (Z_0 - Z_2 + Z_4) / 3  for N=3
_stag_ops  = []; _stag_vals = []
for i, mq in enumerate(range(0, N_QUBITS, 2)):
    p = ['I']*N_QUBITS; p[N_QUBITS-1-mq] = 'Z'
    _stag_ops.append(''.join(p)); _stag_vals.append((-1)**i / N_MATTER)
OBS_STAG     = SparsePauliOp(_stag_ops, _stag_vals)
OBS_STAG_MAT = OBS_STAG.to_matrix()

# Electric flux on link 0–1: X on link qubit 1
# (link qubits are in X-basis so X = diagonal in natural link basis)
_p = ['I']*N_QUBITS; _p[N_QUBITS-1-1] = 'X'
OBS_FLUX     = SparsePauliOp(''.join(_p))
OBS_FLUX_MAT = OBS_FLUX.to_matrix()

PAULI_STAG = str(OBS_STAG.simplify())
print(f'Staggered magnetisation: {OBS_STAG}')
print(f'Electric flux (link 0-1): {\'\'.join(_p)}')


def x_state_to_init_sv(x_state):
    """Initial state |ψ₀⟩ = |0⟩_matter ⊗ |x⟩_links.

    x_state: binary string of length N_LINKS.
    Bit i=0 → link qubit 2i+1 gets X gate (flip to |1⟩).
    Matter qubits always start in |0⟩.
    """
    psi = np.zeros(2**N_QUBITS, dtype=complex); psi[0] = 1.0
    qc_init = QuantumCircuit(N_QUBITS)
    for i, bit in enumerate(reversed(x_state)):   # reversed: bit 0 = link qubit 1
        if bit == '1':
            qc_init.x(2*i+1)   # flip link qubit
    return Statevector(qc_init).data


def exact_obs(x_state, g_list, tau, obs_matrix):
    """⟨O(τ)⟩ under exact matrix-exponentiation."""
    if tau < 1e-9:
        psi = x_state_to_init_sv(x_state)
        return float(np.real(psi.conj() @ obs_matrix @ psi))
    H = H_schwinger(MASS, g_list, H_ELEC).to_matrix()
    psi0 = x_state_to_init_sv(x_state)
    psi_t = expm(1j * H * tau) @ psi0
    return float(np.real(psi_t.conj() @ obs_matrix @ psi_t))


def trotter_obs(x_state, g_list, tau, r, obs_matrix):
    """⟨O(τ)⟩ under first-order Trotter of H_schwinger.

    Per step:
      RZ(2*m*(-1)^i * dt) on each matter qubit  (mass term)
      RX(2*h*dt) on each link qubit              (electric field)
      exp(-i*g_i*dt * X_{m_i}Z_{l_i}X_{m_{i+1}}) via:
          H on m_left, H on m_right → CX(m_left,link)·CX(m_right,link) → RZ(-2*g_i*dt) on link → undo
    """
    if tau < 1e-9:
        psi = x_state_to_init_sv(x_state)
        return float(np.real(psi.conj() @ obs_matrix @ psi))
    dt = tau / r
    qc = QuantumCircuit(N_QUBITS)
    # Initialise gauge links
    for i, bit in enumerate(reversed(x_state)):
        if bit == '1': qc.x(2*i+1)
    for _ in range(r):
        # Mass term: RZ(2*m*(-1)^i*dt) on matter qubits
        for i, mq in enumerate(range(0, N_QUBITS, 2)):
            qc.rz(2.0 * MASS * (-1)**i * dt, mq)
        # Electric field: RX(2*h*dt) on link qubits
        for lq in range(1, N_QUBITS, 2):
            qc.rx(2.0 * H_ELEC * dt, lq)
        # Gauge coupling: g_i * X_{m_i} Z_{l_i} X_{m_{i+1}}
        # Implemented as: H⊗H → CX-ladder → RZ → undo
        for i in range(N_LINKS):
            ml, lk, mr = 2*i, 2*i+1, 2*i+2
            qc.h(ml); qc.h(mr)
            qc.cx(ml, lk); qc.cx(mr, lk)
            qc.rz(2.0 * g_list[i] * dt, lk)
            qc.cx(mr, lk); qc.cx(ml, lk)
            qc.h(ml); qc.h(mr)
    sv = Statevector(qc)
    return float(np.real(sv.data.conj() @ obs_matrix @ sv.data))


# Quick sanity check
print('Sanity check at τ=1.0, x=10, g=[1.0, 0.7]:')
for obs_name, obs_mat in [('M_stag', OBS_STAG_MAT), ('Flux_01', OBS_FLUX_MAT)]:
    e  = exact_obs(TEST_X, TRUE_G, 1.0, obs_mat)
    tr = trotter_obs(TEST_X, TRUE_G, 1.0, R_STEPS, obs_mat)
    print(f'  {obs_name}: exact={e:+.5f}  trotter={tr:+.5f}  err={abs(e-tr):.5f}')

## 3 — Feature extraction via A(U) circuit

The A(U) circuit for the Schwinger model is built by
`CircuitBuilder.build_lgt_trotter_extraction_circuit`, which already implements
the correct structure:
- One frequency register **per gauge link** (d_params = N_LINKS)
- The 3-body Pauli $X_{m_i}Z_{l_i}X_{m_{i+1}}$ is encoded via `apply_pauli_extraction`:
  CNOT from all three qubits in the Pauli support onto the ancilla,
  then V+/V- on `freq_i`

The label function `compute_au_labels_schwinger` maps the physical coupling
$g_i$ to the upload convention: $\pi\alpha_{i,\text{upload}} = g_i\tau/r$.

In [ ]:
builder = CircuitBuilder()

# The LGT circuit uses electric_field as a UNIFORM coupling.
# For spatially-varying g_i we extract features using the existing circuit
# (which encodes the XZX Paulis per link into separate freq registers)
# and provide labels using the full spatially-varying Hamiltonian.
#
# Feature vector from A(U) for gauge state x_state:
#   - One b_l per freq register per link → flattened into one long vector
#   - Shape: (N_LINKS * (4*r+1),) per graph

n_s = builder._freq_register_size(R_STEPS)
SPECTRUM_SIZE = 4 * R_STEPS + 1   # per link register
FEATURE_SIZE  = N_LINKS * SPECTRUM_SIZE

print(f'A(U) circuit for Schwinger model:')
print(f'  n_s = {n_s} qubits per freq register')
print(f'  Spectrum per link: l ∈ [-{2*R_STEPS}, +{2*R_STEPS}]  ({SPECTRUM_SIZE} modes)')
print(f'  N_LINKS = {N_LINKS}  →  total feature vector size = {FEATURE_SIZE}')

# Build example circuit to show structure
qc_demo, qr_freqs_demo = builder.build_lgt_trotter_extraction_circuit(
    num_matter_sites=N_MATTER,
    x_state=TEST_X,
    mass=MASS,
    electric_field=1.0,   # placeholder coupling (same for both links)
    tau=1.0,
    r_steps=R_STEPS,
)
print(f'  Circuit: {qc_demo.num_qubits} qubits  (data={N_QUBITS}, anc=1, freq={N_LINKS}×{n_s})')
print(f'  Gate counts: {dict(qc_demo.count_ops())}')


def extract_b_l_schwinger(x_state, g_upload, tau):
    """Extract Fourier feature vector for gauge state x via A(U) circuit.

    Uses build_lgt_trotter_extraction_circuit with g_upload as the uniform
    coupling placeholder.  In the A(U) encoding, the coupling enters only
    as the Fourier variable (frequency register), so the per-link structure
    is captured correctly by the separate freq registers.

    Parameters
    ----------
    x_state   : binary string, gauge field configuration
    g_upload  : float, upload coupling (= g_phys * tau / (pi * r))
    tau       : float

    Returns
    -------
    np.ndarray, shape (N_LINKS * SPECTRUM_SIZE,)
    """
    qc, qr_freqs = builder.build_lgt_trotter_extraction_circuit(
        num_matter_sites=N_MATTER,
        x_state=x_state,
        mass=MASS,
        electric_field=H_ELEC,
        tau=tau,
        r_steps=R_STEPS,
    )
    sv = Statevector(qc)
    freq_dim = 2 ** n_s
    # data (N_QUBITS) + ancilla (1) qubits occupy the low bits
    da_stride = 2 ** (N_QUBITS + 1)

    b = np.zeros(FEATURE_SIZE)
    for reg_idx in range(N_LINKS):
        # Each freq register occupies n_s bits above the previous registers
        reg_stride = da_stride * (freq_dim ** reg_idx)
        for col, l in enumerate(range(-2*R_STEPS, 2*R_STEPS+1)):
            idx = (l % freq_dim) * reg_stride
            b[reg_idx * SPECTRUM_SIZE + col] = sv.data[idx].real
    return b


def compute_au_labels_schwinger(x_states, g_list, tau, obs_matrix):
    """A(U)-consistent training labels for the Schwinger model.

    Maps physical couplings g_i to upload convention:
        g_i_upload = g_i * tau / (pi * r)
    and builds the Trotter circuit with both mass and electric field fixed.
    """
    labels = np.zeros(len(x_states))
    for idx, x_state in enumerate(x_states):
        labels[idx] = trotter_obs(x_state, g_list, tau, R_STEPS, obs_matrix)
    return labels


# Warm-up
print('\nBuilding gate cache …', end=' ', flush=True)
t0 = time.time()
_ = extract_b_l_schwinger(TEST_X, 1.0, 1.0)
print(f'{time.time()-t0:.1f} s')
t0 = time.time()
_ = extract_b_l_schwinger(TEST_X, 1.0, 1.0)
print(f'Cache hit: {time.time()-t0:.2f} s')

## 4 — Part 1: Staggered magnetisation $\langle M_\mathrm{stag}(t)\rangle$

The staggered magnetisation $\langle M_\mathrm{stag}\rangle = \frac{1}{3}(Z_0 - Z_2 + Z_4)$
is the fermion condensate order parameter — sensitive to both mass and
spatially-varying hopping $g_i$.

In [ ]:
PRECOMPUTED_STAG = '../precomputed_schwinger_stag.npz'
FORCE_RECOMPUTE  = True

if not FORCE_RECOMPUTE and os.path.exists(PRECOMPUTED_STAG):
    _d = np.load(PRECOMPUTED_STAG)
    t_vals = _d['t_vals']
    exact_stag, trotter_stag, pac_stag = _d['exact'], _d['trotter'], _d['pac']

else:
    print('Running M_stag sweep …')
    t_vals = T_VALUES
    exact_stag, trotter_stag, pac_stag = [], [], []
    t_wall = time.time()

    for t in t_vals:
        exact_stag.append(exact_obs(TEST_X, TRUE_G, t, OBS_STAG_MAT))
        trotter_stag.append(trotter_obs(TEST_X, TRUE_G, t, R_STEPS, OBS_STAG_MAT))

        if t < 1e-9:
            pac_stag.append(exact_stag[-1]); continue

        t0 = time.time()
        B_all = np.array([extract_b_l_schwinger(x, 1.0, t) for x in TRAIN_X + [TEST_X]])
        B_train, B_test = B_all[:NUM_TRAIN], B_all[NUM_TRAIN:]
        y_train = compute_au_labels_schwinger(TRAIN_X, TRUE_G, t, OBS_STAG_MAT)

        lasso = Lasso(alpha=LASSO_ALPHA, fit_intercept=False, max_iter=10000)
        lasso.fit(B_train, y_train)
        y_pred = lasso.predict(B_test)[0]
        pac_stag.append(y_pred)

        rank = np.linalg.matrix_rank(B_train, tol=1e-8)
        err  = abs(y_pred - exact_stag[-1])
        print(f'  t={t:.2f}  exact={exact_stag[-1]:+.4f}  '
              f'trotter={trotter_stag[-1]:+.4f}  '
              f'pac={y_pred:+.4f}  |err|={err:.4f}  rank={rank}  ({time.time()-t0:.1f}s)',
              flush=True)

    exact_stag   = np.array(exact_stag)
    trotter_stag = np.array(trotter_stag)
    pac_stag     = np.array(pac_stag)
    np.savez(PRECOMPUTED_STAG, t_vals=t_vals,
             exact=exact_stag, trotter=trotter_stag, pac=pac_stag)
    print(f'\nDone — {time.time()-t_wall:.0f} s')

err_tr_stag  = np.abs(trotter_stag - exact_stag)
err_pac_stag = np.abs(pac_stag     - exact_stag)
print(f'  MAE(Trotter)={np.mean(err_tr_stag):.4f}   MAE(PAC)={np.mean(err_pac_stag):.4f}')

## 5 — Part 2: Electric flux $\langle E_{01}(t)\rangle$

The electric flux on link 0–1 measures the gauge field configuration —
sensitive to the hopping amplitude $g_0$ and the initial gauge state $x$.

In [ ]:
PRECOMPUTED_FLUX = '../precomputed_schwinger_flux.npz'

if not FORCE_RECOMPUTE and os.path.exists(PRECOMPUTED_FLUX):
    _d = np.load(PRECOMPUTED_FLUX)
    t_vals = _d['t_vals']
    exact_flux, trotter_flux, pac_flux = _d['exact'], _d['trotter'], _d['pac']

else:
    print('Running E_flux sweep … (gate cache warm)')
    t_vals = T_VALUES
    exact_flux, trotter_flux, pac_flux = [], [], []
    t_wall = time.time()

    for t in t_vals:
        exact_flux.append(exact_obs(TEST_X, TRUE_G, t, OBS_FLUX_MAT))
        trotter_flux.append(trotter_obs(TEST_X, TRUE_G, t, R_STEPS, OBS_FLUX_MAT))

        if t < 1e-9:
            pac_flux.append(exact_flux[-1]); continue

        t0 = time.time()
        B_all = np.array([extract_b_l_schwinger(x, 1.0, t) for x in TRAIN_X + [TEST_X]])
        B_train, B_test = B_all[:NUM_TRAIN], B_all[NUM_TRAIN:]
        y_train = compute_au_labels_schwinger(TRAIN_X, TRUE_G, t, OBS_FLUX_MAT)

        lasso = Lasso(alpha=LASSO_ALPHA, fit_intercept=False, max_iter=10000)
        lasso.fit(B_train, y_train)
        y_pred = lasso.predict(B_test)[0]
        pac_flux.append(y_pred)

        rank = np.linalg.matrix_rank(B_train, tol=1e-8)
        err  = abs(y_pred - exact_flux[-1])
        print(f'  t={t:.2f}  exact={exact_flux[-1]:+.4f}  '
              f'trotter={trotter_flux[-1]:+.4f}  '
              f'pac={y_pred:+.4f}  |err|={err:.4f}  rank={rank}  ({time.time()-t0:.1f}s)',
              flush=True)

    exact_flux   = np.array(exact_flux)
    trotter_flux = np.array(trotter_flux)
    pac_flux     = np.array(pac_flux)
    np.savez(PRECOMPUTED_FLUX, t_vals=t_vals,
             exact=exact_flux, trotter=trotter_flux, pac=pac_flux)
    print(f'\nDone — {time.time()-t_wall:.0f} s')

err_tr_flux  = np.abs(trotter_flux - exact_flux)
err_pac_flux = np.abs(pac_flux     - exact_flux)
print(f'  MAE(Trotter)={np.mean(err_tr_flux):.4f}   MAE(PAC)={np.mean(err_pac_flux):.4f}')

## 6 — Visualisation

In [ ]:
CLR_EXACT   = '#1f77b4'
CLR_EX2     = '#9467bd'
CLR_TROTTER = '#2ca02c'
CLR_PAC1    = '#d62728'
CLR_PAC2    = '#ff7f0e'

t_dense = np.linspace(0, t_vals[-1], 300)
ex_stag_dense = [exact_obs(TEST_X, TRUE_G, td, OBS_STAG_MAT) for td in t_dense]
ex_flux_dense = [exact_obs(TEST_X, TRUE_G, td, OBS_FLUX_MAT) for td in t_dense]

fig = plt.figure(figsize=(14, 11))
gs  = gridspec.GridSpec(3, 2, figure=fig, hspace=0.52, wspace=0.32)
ax_stag = fig.add_subplot(gs[0, :])
ax_flux = fig.add_subplot(gs[1, :])
ax_err  = fig.add_subplot(gs[2, 0])
ax_feat = fig.add_subplot(gs[2, 1])

# ── Staggered magnetisation ───────────────────────────────────────────────────
ax_stag.plot(t_dense, ex_stag_dense, color=CLR_EXACT, lw=2.0, label=r'Exact $e^{-iHt}$')
ax_stag.plot(t_vals, trotter_stag, color=CLR_TROTTER, lw=1.6, ls='--',
             marker='s', ms=4, label=f'Trotter ($r={R_STEPS}$)')
ax_stag.scatter(t_vals, pac_stag, color=CLR_PAC1, s=60, zorder=5,
                marker='o', edgecolors='white', lw=0.7, label='PAC-Learned')
ax_stag.axhline(0, color='grey', lw=0.5, ls=':')
ax_stag.set_ylabel(r'$\langle M_\mathrm{stag}(t)\rangle$', fontsize=13)
ax_stag.set_title(
    r'Staggered Magnetisation $\langle M_\mathrm{stag}\rangle$ — 1D $\mathbb{Z}_2$ Schwinger Model'
    f'\n($N={N_MATTER}$ matter, $g=[{TRUE_G[0]},{TRUE_G[1]}]$, $m={MASS}$, $h={H_ELEC}$,'
    f' $r={R_STEPS}$, $x_\\mathrm{{test}}$={TEST_X}, {NUM_TRAIN} training states)',
    fontsize=10)
ax_stag.legend(fontsize=9); ax_stag.grid(True, alpha=0.3, ls='--')
ax_stag.set_xlim(-0.05, t_vals[-1]+0.1)

# ── Electric flux ─────────────────────────────────────────────────────────────
ax_flux.plot(t_dense, ex_flux_dense, color=CLR_EX2, lw=2.0, label=r'Exact $e^{-iHt}$')
ax_flux.plot(t_vals, trotter_flux, color=CLR_TROTTER, lw=1.6, ls='--',
             marker='s', ms=4, label=f'Trotter ($r={R_STEPS}$)')
ax_flux.scatter(t_vals, pac_flux, color=CLR_PAC2, s=60, zorder=5,
                marker='D', edgecolors='white', lw=0.7, label='PAC-Learned')
ax_flux.axhline(0, color='grey', lw=0.5, ls=':')
ax_flux.set_ylabel(r'$\langle E_{01}(t)\rangle$', fontsize=13)
ax_flux.set_title(
    r'Electric Flux $\langle E_{01}\rangle$ — Gauge Field on Link 0–1', fontsize=11)
ax_flux.set_xlabel('Time $t$', fontsize=12)
ax_flux.legend(fontsize=9); ax_flux.grid(True, alpha=0.3, ls='--')
ax_flux.set_xlim(-0.05, t_vals[-1]+0.1)

# ── Combined error ────────────────────────────────────────────────────────────
ax_err.semilogy(t_vals, err_tr_stag   + 1e-12, color=CLR_TROTTER, ls='--',
                marker='s', ms=4, alpha=0.6, label=r'Trotter — $M_\mathrm{stag}$')
ax_err.semilogy(t_vals, err_tr_flux   + 1e-12, color='#17becf', ls=':',
                marker='^', ms=4, alpha=0.6, label=r'Trotter — $E_{01}$')
ax_err.semilogy(t_vals, err_pac_stag  + 1e-12, color=CLR_PAC1,
                marker='o', ms=5, label=r'PAC — $M_\mathrm{stag}$')
ax_err.semilogy(t_vals, err_pac_flux  + 1e-12, color=CLR_PAC2,
                marker='D', ms=5, label=r'PAC — $E_{01}$')
ax_err.set_xlabel('Time $t$', fontsize=12)
ax_err.set_ylabel(r'$|\hat{y}-y_\mathrm{exact}|$', fontsize=12)
ax_err.set_title('Absolute Error vs Exact', fontsize=11)
ax_err.legend(fontsize=8); ax_err.grid(True, alpha=0.3, ls='--')
ax_err.set_xlim(-0.05, t_vals[-1]+0.1)

# ── Feature vectors ───────────────────────────────────────────────────────────
cmap = plt.cm.plasma
t_showcase = [0.5, 1.0, 2.0, 3.0]
freq_axis  = np.arange(-2*R_STEPS, 2*R_STEPS+1)
for k, ts in enumerate(t_showcase):
    b_full = extract_b_l_schwinger(TEST_X, 1.0, ts)
    c = cmap(k/(len(t_showcase)-1))
    # Show first freq register (link 0) only for clarity
    b_link0 = b_full[:SPECTRUM_SIZE]
    for xi, yi in zip(freq_axis, b_link0):
        ax_feat.vlines(xi, 0, yi, color=c, lw=2.0, alpha=0.85)
    ax_feat.scatter(freq_axis, b_link0, color=c, s=35, zorder=5)
ax_feat.axhline(0, color='grey', lw=0.6)
ax_feat.set_xlabel('Frequency $l$', fontsize=12)
ax_feat.set_ylabel(r'$\mathrm{Re}(b_l^{(0)}(x_\mathrm{test}))$', fontsize=12)
ax_feat.set_title(r'Feature vector $b_l$ — link 0 freq register', fontsize=11)
handles = [Line2D([0],[0], color=cmap(k/(len(t_showcase)-1)),
                  marker='o', ms=7, label=f'$t={ts}$')
           for k, ts in enumerate(t_showcase)]
ax_feat.legend(handles=handles, fontsize=9)
ax_feat.grid(True, alpha=0.2, ls='--')

plt.savefig('../results_schwinger.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

## 7 — Quantitative summary & discussion

In [ ]:
print('─'*62)
print(f'  {"Observable":<22}  {"Method":<14}  {"MAE":>7}  {"Max |err|":>10}')
print('─'*62)
for name, err_tr, err_pac in [
    ('M_stag (condensate)', err_tr_stag, err_pac_stag),
    ('E_flux (link 0-1)',   err_tr_flux, err_pac_flux),
]:
    print(f'  {name:<22}  {"Trotter":<14}  {np.mean(err_tr):7.4f}  {np.max(err_tr):10.4f}')
    print(f'  {"":<22}  {"PAC-Learned":<14}  {np.mean(err_pac):7.4f}  {np.max(err_pac):10.4f}')
    print('─'*62)

print()
print('Feature matrix properties at t=1.0 (M_stag):')
B = np.array([extract_b_l_schwinger(x, 1.0, 1.0) for x in TRAIN_X])
rank = np.linalg.matrix_rank(B, tol=1e-8)
sv   = np.linalg.svd(B, compute_uv=False)
print(f'  Train samples  : {len(TRAIN_X)} (all {2**N_LINKS} gauge states minus test)')
print(f'  Feature shape  : {B.shape}')
print(f'  Rank           : {rank}')
print(f'  Condition      : {sv[0]/sv[rank-1]:.0f}')
print()
print('Comparison across models:')
print(f'  TFIM         d=1,    {2**3-1} train graphs,  feature rank ~5')
print(f'  Model 4      d=2,    6 train graphs,    feature rank ~5')
print(f'  Schwinger    d={N_LINKS},    {NUM_TRAIN} gauge states,  feature rank={rank}')
print()
print('Key question from the slides:')
print('  Does the Fourier spectrum remain SPARSE for the Schwinger model?')
print('  If yes → BRGD25 works despite d=poly(n)')
print('  If no  → method fails, confirming the d=O(log n) requirement')